# Test evaluation and single-image demo

Run this notebook only after training both models. It evaluates once on the original, untouched test folders and compares accuracy, macro precision/recall/F1, per-class scores, and confusion matrices.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset import get_dataset_splits
from src.torch_dataset import OrangeImageDataset
from src.classical_model import extract_feature_matrix
from src.dl_model import build_orange_classifier
from main_demo import predict_image

splits = get_dataset_splits(PROJECT_ROOT / 'data' / 'raw', validation_size=0.15, random_state=42)
class_names = splits['class_names']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classical_model_path = PROJECT_ROOT / 'models' / 'orange_classical.joblib'
checkpoint_path = PROJECT_ROOT / 'models' / 'orange_mobilenet_v2.pth'
if not classical_model_path.is_file() or not checkpoint_path.is_file():
    raise FileNotFoundError('Run the training cells in notebooks 2 and 3 before evaluation.')
classical_model = joblib.load(classical_model_path)
checkpoint = torch.load(checkpoint_path, map_location=device)
if checkpoint['class_names'] != class_names:
    raise ValueError('Saved model class order does not match the dataset folders.')
dl_model = build_orange_classifier(len(class_names), pretrained=False).to(device)
dl_model.load_state_dict(checkpoint['model_state_dict'])
dl_model.eval()

In [ ]:
X_test, y_test = extract_feature_matrix(splits['test'])
classical_predictions = classical_model.predict(X_test)

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
test_dataset = OrangeImageDataset(splits['test'], class_names, test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
dl_predictions, dl_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = dl_model(images.to(device))
        dl_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        dl_labels.extend(labels.tolist())
dl_predictions = [class_names[index] for index in dl_predictions]
y_test = list(y_test)

for model_name, predictions in [('Classical SVM', classical_predictions), ('MobileNetV2', dl_predictions)]:
    print('\n' + model_name)
    print(classification_report(y_test, predictions, labels=class_names, zero_division=0))
    print('Confusion matrix (rows=true, columns=predicted):')
    display(pd.DataFrame(confusion_matrix(y_test, predictions, labels=class_names), index=class_names, columns=class_names))

metric_rows = []
for model_name, predictions in [('Classical SVM', classical_predictions), ('MobileNetV2', dl_predictions)]:
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average='macro', zero_division=0)
    metric_rows.append({'model': model_name, 'accuracy': accuracy_score(y_test, predictions), 'macro_precision': precision, 'macro_recall': recall, 'macro_f1': f1})
metrics = pd.DataFrame(metric_rows).set_index('model')
display(metrics.round(4))

import json
import platform
import platform
import time

report_dir = PROJECT_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
results = {'dataset': 'orange image dataset', 'test_split': 'provided data/raw/test', 'test_samples': len(y_test), 'class_order': class_names, 'models': {}}
prediction_rows = []
for model_name, predictions in [('Classical SVM', classical_predictions), ('MobileNetV2', dl_predictions)]:
    model_report = classification_report(y_test, predictions, labels=class_names, output_dict=True, zero_division=0)
    model_matrix = confusion_matrix(y_test, predictions, labels=class_names)
    results['models'][model_name] = {
        'accuracy': float(accuracy_score(y_test, predictions)),
        'macro_precision': float(model_report['macro avg']['precision']),
        'macro_recall': float(model_report['macro avg']['recall']),
        'macro_f1': float(model_report['macro avg']['f1-score']),
        'per_class': {name: model_report[name] for name in class_names},
        'confusion_matrix': model_matrix.tolist(),
    }
    prediction_rows.extend({'model': model_name, 'image': str(path.relative_to(PROJECT_ROOT)), 'true_label': truth, 'prediction': prediction} for (path, _), truth, prediction in zip(splits['test'], y_test, predictions))

benchmark_batch, _ = next(iter(test_loader))
benchmark_image = benchmark_batch[:1].to(device)
with torch.inference_mode():
    for _ in range(5):
        dl_model(benchmark_image)
    latency_samples_ms = []
    for _ in range(30):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        started = time.perf_counter()
        dl_model(benchmark_image)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        latency_samples_ms.append((time.perf_counter() - started) * 1000)
results['mobile_model_inference_benchmark'] = {
    'device': str(device),
    'processor': platform.processor() or platform.machine(),
    'batch_size': 1,
    'warmup_runs': 5,
    'timed_runs': len(latency_samples_ms),
    'input_shape': list(benchmark_image.shape),
    'mean_ms': float(sum(latency_samples_ms) / len(latency_samples_ms)),
    'median_ms': float(pd.Series(latency_samples_ms).median()),
    'p95_ms': float(pd.Series(latency_samples_ms).quantile(0.95)),
    'includes_image_decode_and_resize': False,
}
(report_dir / 'test_metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
pd.DataFrame(prediction_rows).to_csv(report_dir / 'test_predictions.csv', index=False)
print('Saved test metrics and predictions to', report_dir)
print('MobileNetV2 inference-only latency (ms):', results['mobile_model_inference_benchmark'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for axis, title, predictions in zip(axes, ['Classical SVM', 'MobileNetV2'], [classical_predictions, dl_predictions]):
    sns.heatmap(confusion_matrix(y_test, predictions, labels=class_names), annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axis)
    axis.set(title=title, xlabel='Predicted class', ylabel='True class')
    axis.tick_params(axis='x', rotation=30)
plt.tight_layout()
figure_dir = PROJECT_ROOT / 'reports' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / 'confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()

# Demonstrate both trained classifiers on one held-out image.
demo_path, true_label = splits['test'][0]
classical_prediction, dl_prediction = predict_image(demo_path, classical_model, dl_model, class_names, device)
print('Image:', demo_path, '| true label:', true_label)
print('Classical SVM:', classical_prediction)
print('MobileNetV2:', dl_prediction)

## Interpretation and assignment answers

Use the measured test results above; do not copy performance or latency claims from a different dataset or hardware. In production, performance depends on camera, lighting, hardware, and the distribution of incoming fruit. Key limitations are the small held-out test set, label ambiguity, class imbalance, and possible background/lighting shortcuts. A stronger follow-up would collect more balanced data across farms, cultivars, cameras, and lighting, use external validation, and measure throughput on deployment hardware. Most work is expected in data quality inspection, feature/training experiments, and model comparison; replace this estimate with your actual time spent.